# Fold 2 untouched-2022 role validation

## tl;dr

RB carry: FAILS_FOLD_2_POINT_GATES with 49 alerts and 64.1% precision. RB opportunity: FAILS_FOLD_2_POINT_GATES. WR: FAILS_FOLD_2_POINT_GATES. TE: INSUFFICIENT_EVIDENCE. No family passes Fold 2. The 2022 execution was run once from the frozen candidate and was not used to tune a replacement.

## Context & Methods

This notebook is the reader-facing companion to `outputs/role_validation/fold_2/FOLD_2_REPORT.md`. It loads the immutable machine-readable artifacts produced by the single 2022 execution.

### Key Assumptions

- Candidate SHA-256 is fixed before Fold 2.
- Baselines reset within 2022 and end before confirmation.
- Outcomes are the next two qualifying games after the alert.
- Confirmed partial games are excluded; suspected cases remain primary.
- Every comparator is equal-volume within role-family week.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT = ROOT / 'outputs' / 'role_validation' / 'fold_2'
assert OUTPUT.exists(), OUTPUT
pd.set_option('display.max_columns', 30)

## Data

### 1. Verify frozen inputs and data quality

In [2]:
fingerprint = json.loads((OUTPUT / 'frozen_config_fingerprint.json').read_text(encoding='utf-8'))
audit = pd.read_csv(OUTPUT / 'data_audit_2022.csv')
print('Candidate SHA-256:', fingerprint['config_sha256'])
print('Matches Fold 1 report:', fingerprint['config_matches_fold1_report'])
audit

Candidate SHA-256: 4dcf389a1f8fcdd11a9277305a8372fadaabaa830185e07eff5d8fbb274a81c7
Matches Fold 1 report: True


,season,canonical_rows,unique_players,played_games,observed_weeks,duplicate_key_rows,duplicate_key_rate,required_null_cells,required_null_rows,identity_resolved_rows,identity_coverage,quality_pass_rows,quality_pass_rate,qualifying_rows,qualifying_rate
0,2022,7478,555,271,18,0,0.0,0,0,7478,1.0,7478,1.0,7478,1.0


## Results

### 2. Inspect family/method performance and uncertainty

In [3]:
methods = pd.read_csv(OUTPUT / 'family_method_results_2022.csv')
primary_methods = methods.loc[methods['partial_policy'].eq('PRIMARY_CONFIRMED_EXCLUDED')]
primary_methods[['role_family','method','alerts','evaluable_alerts','precision','precision_ci_low','precision_ci_high','reversion_rate','median_retention']]

,role_family,method,alerts,evaluable_alerts,precision,precision_ci_low,precision_ci_high,reversion_rate,median_retention
16,rb_carry_share,full_propwar,49,39,0.641026,0.487179,0.794872,0.150000,0.616781
17,rb_carry_share,naive_spike,49,38,0.473684,0.315789,0.631579,0.400000,0.388108
18,rb_carry_share,normal_game_trend,49,38,0.578947,0.421053,0.736842,0.175000,0.564341
19,rb_carry_share,two_week_raw,49,36,0.555556,0.388889,0.722222,0.230769,0.537915
20,rb_opportunity_share,full_propwar,59,47,0.617021,0.468085,0.744681,0.140000,0.687722
21,rb_opportunity_share,naive_spike,59,45,0.533333,0.377778,0.666667,0.319149,0.562135
22,rb_opportunity_share,normal_game_trend,59,44,0.590909,0.454545,0.727273,0.250000,0.606298
23,rb_opportunity_share,two_week_raw,59,45,0.600000,0.444444,0.733333,0.229167,0.551064
24,te_target_share,full_propwar,4,3,0.000000,0.000000,0.000000,0.666667,-0.049826
25,te_target_share,naive_spike,4,3,0.333333,0.000000,1.000000,0.666667,-0.129530


### 3. Apply locked release gates literally

In [4]:
gates = pd.read_csv(OUTPUT / 'release_gate_results_2022.csv')
gates[['role_family','status','alerts','evaluable_alerts','precision','precision_improvement','reversion_rate','reversion_improvement','median_retention','failed_checks']]

,role_family,status,alerts,evaluable_alerts,precision,precision_improvement,reversion_rate,reversion_improvement,median_retention,failed_checks
0,rb_carry_share,FAILS_FOLD_2_POINT_GATES,49,39,0.641026,0.167341,0.150000,0.250000,0.616781,min_holdout_alerts
1,rb_opportunity_share,FAILS_FOLD_2_POINT_GATES,59,47,0.617021,0.083688,0.140000,0.179149,0.687722,min_absolute_improvement_vs_naive | direction_...
2,wr_target_share,FAILS_FOLD_2_POINT_GATES,30,22,0.363636,0.272727,0.250000,0.268519,0.443551,min_holdout_alerts | min_persistence_precision...
3,te_target_share,INSUFFICIENT_EVIDENCE,4,3,0.000000,-0.333333,0.666667,0.000000,-0.049826,min_holdout_alerts | min_persistence_precision...


### 4. Compare redeveloped 2021 with untouched 2022

In [5]:
generalization = pd.read_csv(OUTPUT / 'generalization_2021_vs_2022.csv')
generalization[['role_family','development_2021_full_alerts','untouched_2022_full_alerts','development_2021_full_precision','untouched_2022_full_precision','delta_2022_minus_2021_full_precision','development_2021_precision_improvement','untouched_2022_precision_improvement','generalization_classification']]

,role_family,development_2021_full_alerts,untouched_2022_full_alerts,development_2021_full_precision,untouched_2022_full_precision,delta_2022_minus_2021_full_precision,development_2021_precision_improvement,untouched_2022_precision_improvement,generalization_classification
0,rb_carry_share,56,49,0.744186,0.641026,-0.103160,0.256381,0.167341,MATERIAL_DETERIORATION
1,rb_opportunity_share,77,59,0.672727,0.617021,-0.055706,0.145455,0.083688,STABLE_GENERALIZATION
2,te_target_share,4,4,0.500000,0.000000,-0.500000,0.500000,-0.333333,INSUFFICIENT_SAMPLE
3,wr_target_share,26,30,0.562500,0.363636,-0.198864,0.386029,0.272727,INSUFFICIENT_SAMPLE


### 5. Check direction and seasonal stability

In [6]:
direction = pd.read_csv(OUTPUT / 'direction_results_2022.csv')
blocks = pd.read_csv(OUTPUT / 'season_block_results_2022.csv')
display(direction.loc[direction['partial_policy'].eq('PRIMARY_CONFIRMED_EXCLUDED') & direction['method'].eq('full_propwar'), ['role_family','direction','alerts','evaluable_alerts','precision','reversion_rate','median_retention']])
display(blocks.loc[blocks['partial_policy'].eq('PRIMARY_CONFIRMED_EXCLUDED') & blocks['method'].eq('full_propwar'), ['role_family','week_block','alerts','evaluable_alerts','precision','reversion_rate','median_retention']])

,role_family,direction,alerts,evaluable_alerts,precision,reversion_rate,median_retention
30,rb_carry_share,decrease,27,21,0.476190,0.238095,0.495693
31,rb_carry_share,increase,22,18,0.833333,0.052632,0.791246
38,rb_opportunity_share,decrease,32,24,0.541667,0.185185,0.595091
39,rb_opportunity_share,increase,27,23,0.695652,0.086957,0.844571
46,te_target_share,decrease,2,2,0.000000,0.500000,-0.076951
47,te_target_share,increase,2,1,0.000000,1.000000,0.187528
52,wr_target_share,decrease,10,6,0.666667,0.111111,0.576226
53,wr_target_share,increase,20,16,0.250000,0.315789,0.383032


,role_family,week_block,alerts,evaluable_alerts,precision,reversion_rate,median_retention
44,rb_carry_share,weeks_13_18,19,11,0.636364,0.166667,0.741721
45,rb_carry_share,weeks_1_6,6,5,0.600000,0.000000,0.550887
46,rb_carry_share,weeks_7_12,24,23,0.652174,0.173913,0.616781
56,rb_opportunity_share,weeks_13_18,26,16,0.500000,0.294118,0.566872
57,rb_opportunity_share,weeks_1_6,6,6,0.833333,0.000000,0.654468
58,rb_opportunity_share,weeks_7_12,27,25,0.640000,0.074074,0.844571
68,te_target_share,weeks_13_18,2,1,0.000000,1.000000,0.187528
69,te_target_share,weeks_7_12,2,2,0.000000,0.500000,-0.076951
76,wr_target_share,weeks_13_18,19,12,0.416667,0.235294,0.449648
77,wr_target_share,weeks_1_6,1,1,1.000000,0.000000,0.708459


### 6. Verify partial-game sensitivity and equal volume

In [7]:
sensitivity = pd.read_csv(OUTPUT / 'family_comparisons_2022.csv')
equal_volume = pd.read_csv(OUTPUT / 'equal_volume_verification_2022.csv')
print('Equal-volume cells:', len(equal_volume))
print('All equal volume:', bool(equal_volume['equal_volume'].all()))
sensitivity[['partial_policy','role_family','full_alerts','full_evaluable_alerts','full_precision','precision_improvement','full_reversion_rate','full_median_retention']]

Equal-volume cells: 216
All equal volume: True


,partial_policy,role_family,full_alerts,full_evaluable_alerts,full_precision,precision_improvement,full_reversion_rate,full_median_retention
0,ALL_INCLUDED,rb_carry_share,49,39,0.641026,0.181566,0.170732,0.616781
1,ALL_INCLUDED,rb_opportunity_share,59,47,0.617021,0.071567,0.140000,0.687722
2,ALL_INCLUDED,te_target_share,4,3,0.000000,-0.333333,0.666667,-0.049826
3,ALL_INCLUDED,wr_target_share,30,22,0.363636,0.268398,0.250000,0.443551
4,PRIMARY_CONFIRMED_EXCLUDED,rb_carry_share,49,39,0.641026,0.167341,0.150000,0.616781
5,PRIMARY_CONFIRMED_EXCLUDED,rb_opportunity_share,59,47,0.617021,0.083688,0.140000,0.687722
6,PRIMARY_CONFIRMED_EXCLUDED,te_target_share,4,3,0.000000,-0.333333,0.666667,-0.049826
7,PRIMARY_CONFIRMED_EXCLUDED,wr_target_share,30,22,0.363636,0.272727,0.250000,0.443551
8,STRICT_SUSPECTED_EXCLUDED,rb_carry_share,40,31,0.709677,0.073314,0.093750,0.735558
9,STRICT_SUSPECTED_EXCLUDED,rb_opportunity_share,48,37,0.756757,0.117868,0.073171,0.784409


## Takeaways

- RB carry is encouraging but fails because it has 49 rather than 50 alerts; its lift interval crosses zero.
- RB opportunity misses the locked 10-point naive-improvement gate and direction consistency.
- WR does not support automated use; TE remains insufficient.
- No 2022-based redevelopment occurred, and no family is validated.